# calendar

`calendar` 用于生成和分析日历数据，适合月历展示、闰年判断、按周遍历日期、统计工作日和生成简单 HTML 日历。它只处理日历结构，不负责时区、时间点和夏令时；这些场景应结合 `datetime` 或 `zoneinfo`。


In [ ]:
import calendar
from datetime import date


def month_overview(year: int, month: int) -> dict[str, int | bool]:
    first_weekday, days_in_month = calendar.monthrange(year, month)
    return {
        'first_weekday': first_weekday,
        'days_in_month': days_in_month,
        'is_leap_year': calendar.isleap(year),
    }


print(calendar.month(2026, 5))
print('Overview:', month_overview(2026, 5))
print('Leap days 2020-2030:', calendar.leapdays(2020, 2031))
print('Today weekday:', calendar.day_name[date.today().weekday()])


## 按周遍历月份

`Calendar.monthdatescalendar()` 会返回完整周矩阵，矩阵中可能包含上个月或下个月的日期。这个结构很适合渲染日历组件或做按周统计。


In [ ]:
import calendar as calendar_tools
from datetime import date


def month_weeks(year: int, month: int, first_weekday: int = calendar_tools.MONDAY) -> list[list[date]]:
    cal = calendar_tools.Calendar(firstweekday=first_weekday)
    return cal.monthdatescalendar(year, month)


for week in month_weeks(2026, 5):
    print([day.isoformat() for day in week])


## 工作日统计

`itermonthdates()` 适合按日期流式遍历一个月。统计工作日时要过滤掉矩阵中属于相邻月份的日期，再排除周末和节假日。


In [ ]:
import calendar as business_calendar
from datetime import date


def business_days(year: int, month: int, holidays: set[date] | None = None) -> list[date]:
    holidays = holidays or set()
    cal = business_calendar.Calendar(firstweekday=business_calendar.MONDAY)
    days: list[date] = []

    for day in cal.itermonthdates(year, month):
        if day.month != month:
            continue
        if day.weekday() >= 5:
            continue
        if day in holidays:
            continue
        days.append(day)

    return days


may_holidays = {date(2026, 5, 1)}
workdays = business_days(2026, 5, may_holidays)
print('Business days:', len(workdays))
print('First five:', workdays[:5])


## 文本和 HTML 输出

`TextCalendar` 适合命令行输出，`HTMLCalendar` 可以快速生成简单 HTML 表格。生产环境里通常会把这些数据喂给模板或前端组件，而不是直接使用默认样式。


In [ ]:
import calendar as html_calendar

text_calendar = html_calendar.TextCalendar(firstweekday=html_calendar.SUNDAY)
html_month = html_calendar.HTMLCalendar(firstweekday=html_calendar.SUNDAY).formatmonth(2026, 5)

print(text_calendar.formatmonth(2026, 5))
print(html_month[:200])
